# SAM-Med3D + nnU-Net — Demo pada Data RSHS (format .nii)

Notebook ini untuk **mencoba model pada data RSHS**. Dua jalur disediakan:

- **MODE A — ADA GT**: data RSHS yang kebetulan punya label (untuk validasi). ROI diambil dari centroid GT, DSC & kontur hijau tampil, sama seperti pipeline paper.
- **MODE B — TANPA GT**: data klinis mentah tanpa label. Pusat ROI **ditentukan manual lewat klik**, prediksi saja (tanpa DSC).

Plus: **nnU-Net otomatis** pada .nii yang sama, dan **ekspor hasil ke .nii**.

**Catatan jujur (sampaikan ke beliau):**
- Kedua model dilatih di CMRxMotion. Di scanner RSHS ada **domain shift** — nnU-Net (otomatis, tak bisa dikoreksi) kemungkinan lebih lemah; justru di sinilah nilai SAM-Med3D: klik bisa mengoreksi. Jangan dijanjikan akurasinya setara.
- .nii klinis sering beda orientasi; notebook memakai orientasi kanonik agar irisan tampil benar. Hasil ekspor sebaiknya dibuka di viewer sadar-koordinat (3D Slicer / ITK-SNAP) agar menempel pas.

## Langkah 0 — Prasyarat

Sama seperti notebook setup (GPU, `torch`, `nibabel`, `scipy`, `matplotlib`, `ipywidgets`, `ipympl`, repo SAM-Med3D + checkpoint). **Tambahan untuk nnU-Net**: paket `nnunetv2` terpasang dan folder model nnU-Net terlatih tersedia (`nnUNet_results`).

Semua input/output memakai **.nii / .nii.gz**.

## Langkah 1 — Konfigurasi

**Taruh data baru di folder `D:\Joel\RSHS\`** (buat foldernya kalau belum ada), misalnya:

```
D:\Joel\RSHS\
   ├─ case01.nii.gz          # citra (wajib), 3D atau 4D cine (4D -> frame pertama)
   └─ case01-label.nii.gz    # label (opsional, hanya untuk MODE A)
```

Lalu ubah `RSHS_IMAGE` di sel bawah ke nama file citranya.
- **MODE B (tanpa GT)**: biarkan `RSHS_LABEL = None`.
- **MODE A (ada GT)**: isi `RSHS_LABEL = Path(r"D:\Joel\RSHS\case01-label.nii.gz")`. Label harus satu geometri dengan citra dan memakai kode kelas **1=LV, 2=MYO, 3=RV** (sama seperti CMRxMotion).

Semua hasil (PNG, prediksi .nii.gz) disimpan di `D:\Joel\work\results_rshs\`.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, FileLink
from scipy import ndimage
import torch, torch.nn.functional as F
%matplotlib widget

# --- path model (sama seperti pipeline paper) ---
SAM_REPO      = Path(r"D:\Joel\SAM-Med3D")
WORK_DIR      = Path(r"D:\Joel\work")
RESULT_DIR    = WORK_DIR/"results_rshs"; RESULT_DIR.mkdir(parents=True, exist_ok=True)  # hasil data baru dipisah dari hasil paper
CKPT_ZEROSHOT = SAM_REPO/"ckpt"/"sam_med3d_turbo.pth"
ckpt_fold     = lambda f: WORK_DIR/"sam_ft_frozen"/f"fold{f}"/"sam_model_best.pth"
SAM_SPACING=(1.5,1.5,1.5); SAM_ROI=128
CLASS_NAMES={1:"LV",2:"MYO",3:"RV"}; FOCUS={1:"LV",2:"MYO",3:"RV"}
DEVICE="cuda" if torch.cuda.is_available() else "cpu"

# --- input data baru (UBAH INI) --- taruh file di D:\Joel\RSHS\
RSHS_IMAGE = Path(r"D:\Joel\RSHS\case01.nii.gz")   # citra, wajib
RSHS_LABEL = None   # None = MODE B (tanpa GT). Isi Path(...) utk MODE A (ada GT).
RSHS_STRUCT = 1     # 1=LV, 2=MYO, 3=RV (utk penamaan; di MODE A juga utk centroid/skor)
FT_FOLD = 0         # checkpoint fine-tuned fold mana (0..4). Turbo tak terpengaruh.

CTR_ISO = None      # diisi oleh picker di MODE B

# cek awal supaya error-nya jelas
assert RSHS_IMAGE.exists(), f"citra tidak ditemukan: {RSHS_IMAGE}"
if RSHS_LABEL is not None:
    RSHS_LABEL = Path(RSHS_LABEL); assert RSHS_LABEL.exists(), f"label tidak ditemukan: {RSHS_LABEL}"
    _si, _sl = nib.load(str(RSHS_IMAGE)).shape[:3], nib.load(str(RSHS_LABEL)).shape[:3]
    assert _si == _sl, f"ukuran citra {_si} != label {_sl}"
for _p in [CKPT_ZEROSHOT, ckpt_fold(FT_FOLD)]:
    assert _p.exists(), f"checkpoint tidak ada: {_p}"
print('device:', DEVICE, '| ada GT:', RSHS_LABEL is not None)


## Langkah 2 — Preprocessing (dipakai dua mode)

Fungsi dasar sama dengan paper (`resample_iso`, `crop_center`, `znorm`, `dsc`), ditambah:
- `load_image_canonical` → baca .nii dengan orientasi kanonik + ambil affine & spacing asli.
- `crop_center_meta` → seperti `crop_center` tapi mengembalikan offset agar bisa dipetakan balik saat ekspor.
- `paste_center` + `resample_to_shape` + `export_mask_nii` → petakan prediksi ROI 128³ kembali ke geometri asli pasien, simpan .nii.

In [ ]:
def _to3d(a):
    # 4D/5D (cine, atau dimensi ekstra berukuran 1) -> ambil frame pertama sampai tersisa 3D
    while a.ndim>3: a=a[...,0]
    return a
def load_vol(p):
    a=nib.load(str(p)).get_fdata(); return _to3d(a)
def load_image_canonical(p):
    im=nib.as_closest_canonical(nib.load(str(p)))   # orientasi konsisten utk tampilan/ekspor
    a=_to3d(im.get_fdata())
    sp=tuple(float(z) for z in im.header.get_zooms()[:3])
    return a, im.affine, sp
def resample_iso(v,sp,is_label=False):
    f=tuple(float(s)/float(d) for s,d in zip(sp,SAM_SPACING)); return ndimage.zoom(v,f,order=0 if is_label else 1)
def crop_center_meta(v,c,size=SAM_ROI):
    # kembalikan juga st (offset) utk inverse mapping saat ekspor
    h=size//2; st=[int(round(x))-h for x in c]; out=np.zeros((size,)*3,v.dtype); s_,d_=[],[]
    for ax in range(3):
        a=max(0,st[ax]); b=min(v.shape[ax],st[ax]+size); s_.append(slice(a,b)); d_.append(slice(a-st[ax],a-st[ax]+(b-a)))
    out[tuple(d_)]=v[tuple(s_)]; return out, st
def paste_center(roi, iso_shape, st):
    # inverse crop_center_meta
    out=np.zeros(iso_shape, roi.dtype); s_,d_=[],[]
    for ax in range(3):
        a=max(0,st[ax]); b=min(iso_shape[ax], st[ax]+roi.shape[ax]); s_.append(slice(a,b)); d_.append(slice(a-st[ax],a-st[ax]+(b-a)))
    out[tuple(s_)]=roi[tuple(d_)]; return out
def resample_to_shape(v, target_shape, order):
    f=tuple(t/float(s) for t,s in zip(target_shape, v.shape)); r=ndimage.zoom(v,f,order=order)
    out=np.zeros(target_shape, r.dtype); sl=tuple(slice(0,min(a,b)) for a,b in zip(target_shape,r.shape)); out[sl]=r[sl]
    return out
def znorm(roi): return (roi-float(roi.mean()))/(float(roi.std())+1e-6)
def dsc(p,r):
    p,r=p>0,r>0; d=p.sum()+r.sum(); return 0.0 if d==0 else float(2*(p&r).sum()/d)
def export_mask_nii(pred_roi_bool, meta, out_path, value=1):
    # meta dari demo: st, iso_shape, native_shape, affine
    pred_iso=paste_center((pred_roi_bool>0).astype(np.uint8), meta['iso_shape'], meta['st'])
    pred_nat=resample_to_shape(pred_iso, meta['native_shape'], order=0).astype(np.uint8)*value
    nib.save(nib.Nifti1Image(pred_nat, meta['affine']), str(out_path)); return out_path


## Langkah 3 — Model manager SAM-Med3D (sama seperti paper)

`load_state` + `set_model` + cache, lalu fungsi encode/decode per-model dan `DualState` **disalin apa adanya** dari demo aslimu (versi yang sudah divalidasi, tanpa pencampuran bobot).

In [ ]:
if str(SAM_REPO) not in sys.path: sys.path.insert(0,str(SAM_REPO))
from segment_anything.build_sam3D import sam_model_registry3D
def load_state(p):
    o=torch.load(p, map_location="cpu", weights_only=False)
    for k in ("model_state_dict","model","state_dict"):
        if isinstance(o,dict) and k in o: return o[k]
    return o
_cache={}
def set_model(source, fold):
    key=(source, fold if source=="finetuned" else -1)
    if key not in _cache:
        cp=CKPT_ZEROSHOT if source=="turbo" else ckpt_fold(fold)
        assert cp.exists(), f"checkpoint tak ada: {cp}"
        m=sam_model_registry3D["vit_b_ori"](checkpoint=None)
        info=m.load_state_dict(load_state(cp),strict=False)
        _cache[key]=(m.to(DEVICE).eval(), len(info.missing_keys), len(info.unexpected_keys))
        print(f"load {key}: hilang/lebih {len(info.missing_keys)}/{len(info.unexpected_keys)}")
    return _cache[key][1], _cache[key][2]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import ipywidgets as W
from IPython.display import display, FileLink

# --- encode/decode eksplisit per-model (bebas efek global sam) ---
def _get_model_obj(source, fold):
    key = (source, fold if source == "finetuned" else -1)
    if key not in _cache:
        set_model(source, fold)  # isi cache dulu
    return _cache[key][0]

@torch.no_grad()
def _encode_with(model, roi_model):
    x = torch.from_numpy(roi_model.astype(np.float32))[None, None].to(DEVICE)
    return model.image_encoder(x)

@torch.no_grad()
def _decode_with(model, img_emb, points, labels, prev_low):
    R = SAM_ROI
    coords = torch.tensor([[[float(a) for a in p] for p in points]], device=DEVICE)
    labs = torch.tensor([labels], dtype=torch.int, device=DEVICE)
    pl = prev_low if prev_low is not None else torch.zeros(1, 1, R//4, R//4, R//4, device=DEVICE)
    sparse, dense = model.prompt_encoder(points=(coords, labs), boxes=None, masks=pl)
    kw = dict(image_embeddings=img_emb, image_pe=model.prompt_encoder.get_dense_pe(),
              sparse_prompt_embeddings=sparse, dense_prompt_embeddings=dense)
    try:
        low, _ = model.mask_decoder(**kw, multimask_output=False)
    except TypeError:
        low, _ = model.mask_decoder(**kw)
    prob = torch.sigmoid(F.interpolate(low, size=(R, R, R), mode="trilinear", align_corners=False))[0, 0]
    return (prob > 0.5).cpu().numpy(), low.detach()

class DualState:
    def __init__(self, mod_t, mod_f, roi_model, gt):
        self.mod_t, self.mod_f = mod_t, mod_f
        self.gt = gt > 0
        self.emb_t = _encode_with(mod_t, roi_model)
        self.emb_f = _encode_with(mod_f, roi_model)
        self.reset()
    def reset(self):
        self.points = []; self.labels = []
        self.pred_t = np.zeros((SAM_ROI,)*3, bool); self.pred_f = np.zeros((SAM_ROI,)*3, bool)
    def _run_one(self, emb, mod, prev_chain):
        prev = None; pred = np.zeros((SAM_ROI,)*3, bool)
        for i in range(len(self.points)):
            pred, prev = _decode_with(mod, emb, self.points[:i+1], self.labels[:i+1], prev)
        return pred, prev
    def _run(self):
        if not self.points:
            self.pred_t = np.zeros((SAM_ROI,)*3, bool); self.pred_f = np.zeros((SAM_ROI,)*3, bool)
            return
        self.pred_t, _ = self._run_one(self.emb_t, self.mod_t, None)
        self.pred_f, _ = self._run_one(self.emb_f, self.mod_f, None)
    def add(self, pt, label):
        self.points.append(tuple(int(x) for x in pt)); self.labels.append(label); self._run()
    def undo(self):
        if self.points: self.points.pop(); self.labels.pop(); self._run()
    def dsc_t(self): return dsc(self.pred_t, self.gt)
    def dsc_f(self): return dsc(self.pred_f, self.gt)



## Langkah 4 — (MODE B saja) pilih pusat ROI secara manual

Jika **tanpa GT**, jalankan sel ini dulu: geser slice ke jantung, **klik sekali** di pusat jantung, lalu tekan **Pakai pusat terpilih**. (Kalau ada GT, lewati — pusat diambil dari centroid GT otomatis.)

In [ ]:
class CenterPicker:
    def __init__(self):
        img,aff,sp=load_image_canonical(RSHS_IMAGE)
        self.iso=resample_iso(img,sp); self.sa=int(np.argmax(sp))
        self.ctr=[s//2 for s in self.iso.shape]
        self.sl=W.IntSlider(min=0,max=self.iso.shape[self.sa]-1,value=self.ctr[self.sa],description="Slice")
        self.b_mid=W.Button(description="Pakai tengah volume")
        self.b_use=W.Button(description="Pakai pusat terpilih",button_style="success")
        self.lbl=W.HTML()
        plt.ioff(); self.fig,self.ax=plt.subplots(figsize=(5,5)); plt.ion()
        self.fig.canvas.header_visible=False
        self.fig.canvas.mpl_connect("button_press_event",self.on_click)
        self.sl.observe(lambda c:self.draw(),names="value")
        self.b_mid.on_click(self.use_mid); self.b_use.on_click(self.use_sel)
        self.ui=W.VBox([W.HBox([self.sl,self.b_mid,self.b_use]),self.lbl,self.fig.canvas]); self.draw()
    def rem(self): return [a for a in range(3) if a!=self.sa]
    def on_click(self,e):
        if e.inaxes!=self.ax or e.xdata is None: return
        rem=self.rem(); self.ctr[self.sa]=self.sl.value
        self.ctr[rem[0]]=int(round(e.ydata)); self.ctr[rem[1]]=int(round(e.xdata)); self.draw()
    def draw(self):
        self.ax.clear(); z=self.sl.value
        self.ax.imshow(np.take(self.iso,z,axis=self.sa),cmap="gray"); rem=self.rem()
        if self.ctr[self.sa]==z: self.ax.plot(self.ctr[rem[1]],self.ctr[rem[0]],"+",c="yellow",ms=14,mew=3)
        self.ax.set_title(f"Set pusat ROI (slice {z})",fontsize=9); self.ax.axis("off")
        self.lbl.value=f"<b>Pusat ROI (iso):</b> {tuple(int(x) for x in self.ctr)}"; self.fig.canvas.draw_idle()
    def use_mid(self,*_):
        global CTR_ISO; self.ctr=[s//2 for s in self.iso.shape]; self.sl.value=self.ctr[self.sa]
        CTR_ISO=tuple(int(x) for x in self.ctr); self.draw(); print("pusat =",CTR_ISO)
    def use_sel(self,*_):
        global CTR_ISO; CTR_ISO=tuple(int(x) for x in self.ctr); print("pusat dipakai =",CTR_ISO)

if RSHS_LABEL is None:
    _picker=CenterPicker(); display(_picker.ui)
else:
    print("MODE A (ada GT): pusat ROI dari centroid GT, picker tidak diperlukan.")


## Langkah 5 — Demo klik SAM-Med3D (Zero-shot vs Fine-tuned)

UI sama seperti 4B. Di **MODE B** tidak ada DSC/kontur hijau (karena tanpa GT). Tombol **Ekspor .nii** menyimpan prediksi (fine-tuned & Turbo) ke geometri asli pasien di `work/results_rshs/`.

> **MODE B**: kalau baru memilih pusat di Langkah 4 setelah sel ini dijalankan, tekan tombol **Muat** agar ROI dipotong ulang di pusat baru.

In [ ]:
class RSHSDualDemo:
    def __init__(self):
        self.has_gt = RSHS_LABEL is not None
        self.tg_mode=W.ToggleButtons(options=[("Positif +",1),("Negatif −",0)],value=1)
        self.sl_z=W.IntSlider(min=0,max=SAM_ROI-1,value=SAM_ROI//2,description="Slice")
        self.cb_gt=W.Checkbox(value=self.has_gt,description="GT")
        self.b_center=W.Button(description="Klik tengah",button_style="info")
        self.b_undo=W.Button(description="Undo"); self.b_reset=W.Button(description="Reset",button_style="warning")
        self.b_load=W.Button(description="Muat",button_style="primary")
        self.b_png=W.Button(description="Render PNG",button_style="success")
        self.b_nii=W.Button(description="Ekspor .nii",button_style="success")
        self.lbl=W.HTML(); self.out=W.Output()
        plt.ioff(); self.fig,(self.ax_t,self.ax_f)=plt.subplots(1,2,figsize=(10,5)); plt.ion()
        self.fig.canvas.header_visible=False
        self.fig.canvas.mpl_connect("button_press_event",self.on_click)
        for b,fn in [(self.b_load,self.load),(self.b_center,self.center),(self.b_undo,self.undo),
                     (self.b_reset,self.reset),(self.b_png,self.render),(self.b_nii,self.export)]:
            b.on_click(fn)
        self.sl_z.observe(lambda c:self.draw(),names="value"); self.cb_gt.observe(lambda c:self.draw(),names="value")
        self.state=None
        self.ui=W.VBox([W.HBox([self.b_load,self.tg_mode,self.b_center,self.b_undo,self.b_reset,self.cb_gt]),
                        W.HBox([self.sl_z,self.b_png,self.b_nii]),self.lbl,self.fig.canvas,self.out])
        self.load()
    def rem(self): return [a for a in range(3) if a!=self.sa]
    def load(self,*_):
        img,aff,sp=load_image_canonical(RSHS_IMAGE)
        img_iso=resample_iso(img,sp); self.iso_shape=img_iso.shape
        self.native_shape=img.shape; self.affine=aff; self.sa=int(np.argmax(sp))
        if self.has_gt:
            lab,_,_=load_image_canonical(RSHS_LABEL)
            gt_iso=resample_iso((lab==RSHS_STRUCT).astype(np.uint8),sp,is_label=True)
            if gt_iso.sum()==0: self.lbl.value="<b>GT kosong utk struktur ini.</b>"; return
            ctr=ndimage.center_of_mass(gt_iso)
        else:
            ctr = CTR_ISO if CTR_ISO is not None else tuple(s//2 for s in img_iso.shape)
        roi_img,self.st=crop_center_meta(img_iso,ctr); self.disp=roi_img; model=znorm(roi_img)
        self.gt = (crop_center_meta(gt_iso,ctr)[0]>0) if self.has_gt else np.zeros((SAM_ROI,)*3,bool)
        self.ctr_roi=tuple(int(round(x)) for x in ndimage.center_of_mass(self.gt)) if self.gt.any() else (SAM_ROI//2,)*3
        mod_t=_get_model_obj("turbo",FT_FOLD); mod_f=_get_model_obj("finetuned",FT_FOLD)
        self.state=DualState(mod_t,mod_f,model,self.gt)
        self.sl_z.value=int(self.ctr_roi[self.sa]); self.draw()
    def center(self,*_):
        if self.state: self.state.add(self.ctr_roi,1); self.sl_z.value=int(self.ctr_roi[self.sa]); self.draw()
    def undo(self,*_):
        if self.state: self.state.undo(); self.draw()
    def reset(self,*_):
        if self.state: self.state.reset(); self.draw()
    def on_click(self,e):
        if self.state is None or e.inaxes not in (self.ax_t,self.ax_f) or e.xdata is None: return
        rem=self.rem(); pt=[0,0,0]; pt[self.sa]=self.sl_z.value
        pt[rem[0]]=int(round(e.ydata)); pt[rem[1]]=int(round(e.xdata))
        self.state.add(pt,self.tg_mode.value); self.draw()
    def _panel(self,ax,pred,title):
        sa,rem,z=self.sa,self.rem(),self.sl_z.value; ax.clear()
        ax.imshow(np.take(self.disp,z,axis=sa),cmap="gray")
        pm=np.take(pred,z,axis=sa)
        if pm.any(): ax.imshow(np.ma.masked_where(~pm,pm),cmap="autumn",alpha=0.45)
        if self.has_gt and self.cb_gt.value:
            g=np.take(self.gt,z,axis=sa)
            if g.any(): ax.contour(g,levels=[0.5],colors="lime",linewidths=1.2)
        for p,l in zip(self.state.points,self.state.labels):
            if p[sa]==z: ax.plot(p[rem[1]],p[rem[0]],"+" if l else "x",c="cyan" if l else "red",ms=11,mew=2.5)
        ax.set_title(title,fontsize=10); ax.axis("off")
    def _ttl(self,tag,d):
        return f"{tag} — {FOCUS[RSHS_STRUCT]}" + (f" | DSC {d:.3f}" if self.has_gt else "")
    def draw(self):
        if self.state is None: return
        self._panel(self.ax_t,self.state.pred_t,self._ttl("Zero-shot (Turbo)",self.state.dsc_t()))
        self._panel(self.ax_f,self.state.pred_f,self._ttl("Fine-tuned",self.state.dsc_f()))
        n=len(self.state.points)
        extra=(f" ZS {self.state.dsc_t():.3f} / FT {self.state.dsc_f():.3f}" if self.has_gt else "")
        self.lbl.value=f"<b>{RSHS_IMAGE.name} {FOCUS[RSHS_STRUCT]}</b> &nbsp; Klik: {n}{extra}"
        self.fig.canvas.draw_idle()
    def _meta(self): return dict(st=self.st,iso_shape=self.iso_shape,native_shape=self.native_shape,affine=self.affine)
    def export(self,*_):
        if self.state is None: return
        self.out.clear_output(); stem=RSHS_IMAGE.name.replace(".nii.gz","").replace(".nii",""); cn=FOCUS[RSHS_STRUCT]
        pt=RESULT_DIR/f"{stem}_{cn}_FT_pred.nii.gz"; export_mask_nii(self.state.pred_f,self._meta(),pt)
        pz=RESULT_DIR/f"{stem}_{cn}_Turbo_pred.nii.gz"; export_mask_nii(self.state.pred_t,self._meta(),pz)
        with self.out:
            print("tersimpan (geometri asli pasien):"); print(" ",pt); print(" ",pz)
            try: display(FileLink(str(pt))); display(FileLink(str(pz)))
            except Exception as e: print(e)
    def render(self,*_):
        if self.state is None: return
        self.out.clear_output(); stem=RSHS_IMAGE.name.replace(".nii.gz","").replace(".nii",""); cn=FOCUS[RSHS_STRUCT]
        z=self.sl_z.value; n=len(self.state.points)
        fig2,(a1,a2)=plt.subplots(1,2,figsize=(10,5.2))
        for ax,pred,tag in [(a1,self.state.pred_t,"Zero-shot (Turbo)"),(a2,self.state.pred_f,"Fine-tuned")]:
            ax.imshow(np.take(self.disp,z,axis=self.sa),cmap="gray")
            pm=np.take(pred,z,axis=self.sa)
            if pm.any(): ax.imshow(np.ma.masked_where(~pm,pm),cmap="autumn",alpha=0.45)
            if self.has_gt:
                g=np.take(self.gt,z,axis=self.sa)
                if g.any(): ax.contour(g,levels=[0.5],colors="lime",linewidths=1.2)
            rem=self.rem()
            for p,l in zip(self.state.points,self.state.labels):
                if p[self.sa]==z: ax.plot(p[rem[1]],p[rem[0]],"+" if l else "x",c="cyan" if l else "red",ms=11,mew=2.5)
            ax.set_title(tag,fontsize=11,fontweight="bold"); ax.axis("off")
        fig2.suptitle(f"{stem} {cn} — manual {n} klik (RSHS)",fontsize=12,fontweight="bold")
        plt.tight_layout()
        out=RESULT_DIR/f"rshs_{stem}_{cn}_{n}klik_z{z}.png"; fig2.savefig(out,dpi=300,bbox_inches="tight")
        with self.out:
            print("tersimpan:",out)
            try: display(FileLink(str(out)))
            except Exception as e: print(e)
        plt.show()

rshs=RSHSDualDemo(); display(rshs.ui)


## Langkah 6 — nnU-Net otomatis (ada/tanpa GT sama saja)

nnU-Net bekerja otomatis (tanpa klik) dan sudah menghasilkan .nii di geometri asli. Dua cara:

- **Cara 1 — jalankan prediksi** via CLI `nnUNetv2_predict` (butuh `nnunetv2` + model terlatih + env `nnUNet_results`).
- **Cara 2 — muat hasil yang sudah ada** (kalau kamu sudah pernah prediksi): set `NNUNET_OUT` ke file .nii-nya.

Setelah hasil ada, sel viewer menampilkan overlay nnU-Net pada citra (per kelas LV/MYO/RV).

In [ ]:
import shutil, subprocess
# --- konfigurasi nnU-Net (sesuai model terlatih di work\nnunet) ---
NNUNET_RESULTS = WORK_DIR/"nnunet"/"nnUNet_results"  # folder hasil training nnU-Net
NNUNET_DATASET = "Dataset111_CMRxMotion"             # nama/ID dataset nnU-Net
NNUNET_TRAINER = "nnUNetTrainer_100epochs"           # trainer yang dipakai saat training
NNUNET_CONFIG  = "3d_fullres"
NNUNET_FOLDS   = "0"                                  # mis. '0' atau '0 1 2 3 4' (ensemble, lebih lama)
NNUNET_OUT     = None   # Cara 2: isi Path(...) ke hasil .nii yg sudah ada -> lewati prediksi

def _nnunet_exe():
    # pakai exe dari venv kernel ini, supaya tidak tergantung PATH
    exe = Path(sys.executable).parent/("nnUNetv2_predict.exe" if os.name=="nt" else "nnUNetv2_predict")
    return str(exe) if exe.exists() else "nnUNetv2_predict"

def run_nnunet(image_path):
    tmp_in=RESULT_DIR/"_nnunet_in"; tmp_out=RESULT_DIR/"_nnunet_out"
    for d in (tmp_in,tmp_out): shutil.rmtree(d,ignore_errors=True); d.mkdir(parents=True)
    # nnU-Net butuh 3D .nii.gz bernama *_0000; 4D (cine) -> frame pertama, sama seperti sisi SAM
    im=nib.load(str(image_path)); a=np.asanyarray(im.dataobj)
    a=_to3d(a)
    nib.save(nib.Nifti1Image(a, im.affine, im.header), str(tmp_in/"case_0000.nii.gz"))
    env=os.environ.copy(); env["nnUNet_results"]=str(NNUNET_RESULTS)
    env.setdefault("nnUNet_raw", str(WORK_DIR/"nnunet"/"nnUNet_raw"))
    env.setdefault("nnUNet_preprocessed", str(WORK_DIR/"nnunet"/"nnUNet_preprocessed"))
    cmd=[_nnunet_exe(),"-i",str(tmp_in),"-o",str(tmp_out),"-d",NNUNET_DATASET,"-tr",NNUNET_TRAINER,
         "-c",NNUNET_CONFIG,"-f",*NNUNET_FOLDS.split(),"-device","cuda" if torch.cuda.is_available() else "cpu"]
    print("menjalankan:"," ".join(cmd))
    r=subprocess.run(cmd,env=env,capture_output=True,text=True)
    if r.returncode!=0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:]); raise RuntimeError("nnUNetv2_predict gagal (lihat log di atas)")
    res=tmp_out/"case.nii.gz"
    if not res.exists(): return None
    stem=Path(image_path).name.replace(".nii.gz","").replace(".nii","")
    final=RESULT_DIR/f"{stem}_nnUNet_pred.nii.gz"; shutil.copy(str(res),str(final))
    return final

if NNUNET_OUT is not None:
    nn_path=Path(NNUNET_OUT); print("pakai hasil nnU-Net yang ada:",nn_path)
else:
    nn_path=run_nnunet(RSHS_IMAGE); print("hasil nnU-Net:",nn_path)


In [ ]:
# viewer overlay nnU-Net (label map di geometri asli), selaras dgn citra kanonik
class NNViewer:
    def __init__(self, image_path, seg_path):
        self.img,_,sp=load_image_canonical(image_path)
        self.seg=_to3d(nib.as_closest_canonical(nib.load(str(seg_path))).get_fdata()).astype(int)
        self.sa=int(np.argmax(sp))
        self.sl=W.IntSlider(min=0,max=self.img.shape[self.sa]-1,value=self.img.shape[self.sa]//2,description="Slice")
        plt.ioff(); self.fig,self.ax=plt.subplots(figsize=(5.5,5.5)); plt.ion()
        self.fig.canvas.header_visible=False; self.sl.observe(lambda c:self.draw(),names="value")
        self.ui=W.VBox([self.sl,self.fig.canvas]); self.draw()
    def draw(self):
        z=self.sl.value; self.ax.clear()
        self.ax.imshow(np.take(self.img,z,axis=self.sa),cmap="gray")
        seg=np.take(self.seg,z,axis=self.sa)
        for cls,col in [(1,"red"),(2,"yellow"),(3,"cyan")]:
            m=seg==cls
            if m.any(): self.ax.contour(m,levels=[0.5],colors=col,linewidths=1.2)
        self.ax.set_title(f"nnU-Net overlay (LV merah, MYO kuning, RV cyan) z{z}",fontsize=9); self.ax.axis("off")
        self.fig.canvas.draw_idle()

if nn_path: nnv=NNViewer(RSHS_IMAGE, nn_path); display(nnv.ui)
else: print("nnU-Net belum menghasilkan output; cek konfigurasi.")


## Catatan & Troubleshooting

- **Pemilihan fold FT**: model fine-tuned ada 5 (per fold CV di CMRxMotion). Untuk data RSHS eksternal, fold mana pun valid; default `FT_FOLD=0`. Kalau mau lebih kokoh, bandingkan beberapa fold.
- **Domain shift**: hasil pada scanner RSHS bisa beda dari CMRxMotion. SAM-Med3D bisa dikoreksi dengan klik; nnU-Net tidak. Set ekspektasi.
- **Orientasi .nii**: notebook pakai orientasi kanonik. Buka hasil ekspor di 3D Slicer / ITK-SNAP (sadar-koordinat) agar menempel pas pada citra asli.
- **Pusat ROI (MODE B)**: kalau jantung tidak masuk penuh ke kubus 128³ (≈19 cm pada 1.5 mm), geser pusat atau pilih slice lain saat picking.
- **nnU-Net error** `command not found` → `pip install nnunetv2`; `-d/-c/-f` harus cocok dengan model terlatihmu; `nnUNet_results` harus menunjuk folder model. Pesan error lengkap dari nnU-Net tampil di output sel.
- **UI tak muncul** → `ipympl` + restart kernel, pastikan `%matplotlib widget` jalan.